In [7]:
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
data_dir = project_root / "data" / "raw"
print("프로젝트 루트:", project_root)
print("데이터 폴더:", data_dir)
print("데이터 폴더 존재:", data_dir.exists())

프로젝트 루트: c:\dev\ai-data-analysis
데이터 폴더: c:\dev\ai-data-analysis\data\raw
데이터 폴더 존재: True


In [8]:
required_files = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
]

for file_name in required_files:
    file_path = data_dir / file_name
    print(file_name, file_path.exists(), file_path.stat().st_size if file_path.exists() else 0)

customers.csv True 5641
products.csv True 4103
orders.csv True 11655
order_items.csv True 15335


In [9]:
import pandas as pd

customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")

In [10]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

In [11]:
for name, df in datasets.items():
    print(f"\n[{name}]")
    display(df.isna().sum().to_frame("missing_count"))


[customers]


,missing_count
customer_id,0
name,0
gender,0
age,0
city,0
signup_date,0



[products]


,missing_count
product_id,0
product_name,0
category,0
price,0



[orders]


,missing_count
order_id,0
customer_id,0
order_date,0
payment_method,0
order_status,0



[order_items]


,missing_count
order_item_id,0
order_id,0
product_id,0
quantity,0
unit_price,0


In [12]:
for name, df in datasets.items():
    print(name, "전체 중복:", df.duplicated().sum())

customers 전체 중복: 0
products 전체 중복: 0
orders 전체 중복: 0
order_items 전체 중복: 0


In [13]:
primary_keys = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
}
id_check_rows = []
for name, key in primary_keys.items():
    df = datasets[name]
    id_check_rows.append(
        {
            "dataset": name,
            "key": key,
            "missing": df[key].isna().sum(),
            "duplicates": df[key].duplicated().sum(),
            "unique": df[key].nunique(),
            "rows": len(df),
        }
    )
id_check_summary = pd.DataFrame(id_check_rows)
id_check_summary

,dataset,key,missing,duplicates,unique,rows
0,customers,customer_id,0,0,150,150
1,products,product_id,0,0,100,100
2,orders,order_id,0,0,300,300
3,order_items,order_item_id,0,0,764,764


In [15]:

print(customers[["age"]].describe())
print(products[["price"]].describe())
print(order_items[["quantity", "unit_price"]].describe())

              age
count  150.000000
mean    42.086667
std     15.613166
min     19.000000
25%     29.000000
50%     40.000000
75%     57.000000
max     69.000000
               price
count     100.000000
mean   110040.000000
std     56433.910574
min      5000.000000
25%     65750.000000
50%    112000.000000
75%    161000.000000
max    200000.000000
         quantity     unit_price
count  764.000000     764.000000
mean     3.053665  108561.518325
std      1.410873   56996.770604
min      1.000000    5000.000000
25%      2.000000   62000.000000
50%      3.000000  111000.000000
75%      4.000000  161250.000000
max      5.000000  200000.000000


In [16]:
category_columns = {
    "customers": ["gender", "city"],
    "products": ["category"],
    "orders": ["payment_method", "order_status"],
}
for dataset_name, columns in category_columns.items():
    df = datasets[dataset_name]
    for column in columns:
        print(f"\n[{dataset_name}.{column}]")
        display(
            df[column]
            .value_counts(dropna=False)
            .to_frame("count")
        )


[customers.gender]


,count
gender,
F,84
M,66



[customers.city]


,count
city,
성남,21
광주,17
부산,16
대구,15
서울,15
울산,14
인천,14
대전,14
수원,13



[products.category]


,count
category,
스포츠,19
전자기기,17
생활용품,16
뷰티,16
도서,14
패션,11
식품,7



[orders.payment_method]


,count
payment_method,
kakao_pay,79
naver_pay,77
bank_transfer,74
card,70



[orders.order_status]


,count
order_status,
completed,184
cancelled,64
refunded,52


In [17]:
def inspect_dataframe(
    name: str,
    df: pd.DataFrame,
    primary_key: str | None = None,
) -> dict:
    result = {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": df.columns.tolist(),
        "dtypes": {
            column: str(dtype)
            for column, dtype in df.dtypes.items()
        },
        "missing_total": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
    }
    if primary_key is not None:
        result["primary_key"] = primary_key
        result["key_missing"] = int(
            df[primary_key].isna().sum()
        )
        result["key_duplicates"] = int(
            df[primary_key].duplicated().sum()
        )
        result["key_unique"] = int(
            df[primary_key].nunique()
        )
    return result

In [18]:
inspection_results = []

for name, df in datasets.items():
    inspection_results.append(
        inspect_dataframe(
            name,
            df,
            primary_keys[name],
        )
    )
inspection_results

[{'dataset': 'customers',
  'rows': 150,
  'columns': 6,
  'column_names': ['customer_id',
   'name',
   'gender',
   'age',
   'city',
   'signup_date'],
  'dtypes': {'customer_id': 'int64',
   'name': 'str',
   'gender': 'str',
   'age': 'int64',
   'city': 'str',
   'signup_date': 'str'},
  'missing_total': 0,
  'duplicate_rows': 0,
  'primary_key': 'customer_id',
  'key_missing': 0,
  'key_duplicates': 0,
  'key_unique': 150},
 {'dataset': 'products',
  'rows': 100,
  'columns': 4,
  'column_names': ['product_id', 'product_name', 'category', 'price'],
  'dtypes': {'product_id': 'int64',
   'product_name': 'str',
   'category': 'str',
   'price': 'int64'},
  'missing_total': 0,
  'duplicate_rows': 0,
  'primary_key': 'product_id',
  'key_missing': 0,
  'key_duplicates': 0,
  'key_unique': 100},
 {'dataset': 'orders',
  'rows': 300,
  'columns': 5,
  'column_names': ['order_id',
   'customer_id',
   'order_date',
   'payment_method',
   'order_status'],
  'dtypes': {'order_id': 'int6